In [2]:
import fiftyone.zoo as foz
import torch
from ultralytics import YOLO
from PIL import Image
import numpy as np
import fiftyone as fo
import fiftyone.utils.random as four
import fiftyone as fo

In [ ]:
dataset = foz.load_zoo_dataset(
    "coco-2017",
    splits=["train", "validation", "test"],
    #splits=["train"],
    label_types=["detections"],
    classes=["chair"],
)

# Rename the dataset
dataset.name = "der_merger"
dataset.save()

NameError: name 'foz' is not defined

In [4]:
print(dataset)

Name:        coco-2017-train-validation-test-1000
Media type:  image
Num samples: 2580
Persistent:  False
Tags:        []
Sample fields:
    id:               fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField
    ground_truth:     fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Detections)


In [5]:
dataset.compute_metadata()

In [6]:


# Get class list
classes = dataset.default_classes

# Run the model on GPU if it is available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Load the YOLO model (not just the path)
model = YOLO("/mnt/EBI-SHARE/06_Data/labelstudio/models/doors/best.pt")
model.to(device)

print("Model ready")

# Add predictions to samples
with fo.ProgressBar() as pb:
    for sample in pb(dataset):
        # Load image as PIL Image (don't convert to tensor)
        image = Image.open(sample.filepath)
        
        # Perform inference directly on PIL image
        results = model(image, conf=0.4)
        
        # Extract predictions from results
        result = results[0]
        boxes = result.boxes.xyxy.cpu().numpy()  # x1, y1, x2, y2
        scores = result.boxes.conf.cpu().numpy()
        labels = result.boxes.cls.cpu().numpy().astype(int)
        
        # Get image dimensions
        w, h = image.size
        
        # Convert detections to FiftyOne format
        detections = []
        for label, score, box in zip(labels, scores, boxes):
            # Convert to [top-left-x, top-left-y, width, height]
            # in relative coordinates in [0, 1] x [0, 1]
            x1, y1, x2, y2 = box
            rel_box = [x1 / w, y1 / h, (x2 - x1) / w, (y2 - y1) / h]

            detections.append(
                fo.Detection(
                    label=model.names[label],  # Use model's class names
                    bounding_box=rel_box,
                    confidence=score
                )
            )
        # Save predictions to dataset
        sample["predictions"] = fo.Detections(detections=detections)
        sample.save()

# Filter samples that have "Door" predictions
door_predictions_view = dataset.filter_labels("predictions", fo.ViewField("label") == "Door")

# Get the sample IDs that contain Door predictions
door_sample_ids = list(door_predictions_view.values("id"))

# Delete these samples from the dataset
# dataset.delete_samples(door_sample_ids)

print(f"Deleted {len(door_sample_ids)} samples with Door predictions")

# Save the changes
dataset.save()

Model ready
                                                                             
0: 480x640 (no detections), 29.5ms
Speed: 1.9ms preprocess, 29.5ms inference, 11.5ms postprocess per image at shape (1, 3, 480, 640)
                                                                                 
0: 640x416 (no detections), 30.7ms
Speed: 1.0ms preprocess, 30.7ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 416)

0: 480x640 (no detections), 6.8ms
Speed: 1.1ms preprocess, 6.8ms inference, 0.8ms postprocess per image at shape (1, 3, 480, 640)

0: 640x448 (no detections), 29.8ms
Speed: 0.7ms preprocess, 29.8ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 448)
                                                                                 
0: 448x640 (no detections), 29.7ms
Speed: 0.9ms preprocess, 29.7ms inference, 0.5ms postprocess per image at shape (1, 3, 448, 640)

0: 480x640 (no detections), 6.9ms
Speed: 1.9ms preprocess, 6.9ms inference, 0.4ms po

In [1]:
# Visualize the dataset in the FiftyOne App

session = fo.launch_app(dataset)

NameError: name 'fo' is not defined

In [6]:


# Get class list
classes = dataset.default_classes

# Run the model on GPU if it is available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Load the YOLO model (not just the path)
model = YOLO("/mnt/EBI-SHARE/06_Data/labelstudio/models/windows/best.pt")
model.to(device)

print("Model ready")

# Add predictions to samples
with fo.ProgressBar() as pb:
    for sample in pb(dataset):
        # Load image as PIL Image (don't convert to tensor)
        image = Image.open(sample.filepath)
        
        # Perform inference directly on PIL image
        results = model(image, conf=0.4)
        
        # Extract predictions from results
        result = results[0]
        boxes = result.boxes.xyxy.cpu().numpy()  # x1, y1, x2, y2
        scores = result.boxes.conf.cpu().numpy()
        labels = result.boxes.cls.cpu().numpy().astype(int)
        
        # Get image dimensions
        w, h = image.size
        
        # Convert detections to FiftyOne format
        detections = []
        for label, score, box in zip(labels, scores, boxes):
            # Convert to [top-left-x, top-left-y, width, height]
            # in relative coordinates in [0, 1] x [0, 1]
            x1, y1, x2, y2 = box
            rel_box = [x1 / w, y1 / h, (x2 - x1) / w, (y2 - y1) / h]

            detections.append(
                fo.Detection(
                    label=model.names[label],  # Use model's class names
                    bounding_box=rel_box,
                    confidence=score
                )
            )

        # Save predictions to dataset
        sample["predictions"] = fo.Detections(detections=detections)
        sample.save()

# Filter samples that have "Window" predictions
window_predictions_view = dataset.filter_labels("predictions", fo.ViewField("label") == "window")

# Get the sample IDs that contain Window predictions
window_sample_ids = list(window_predictions_view.values("id"))

# Delete these samples from the dataset
dataset.delete_samples(window_sample_ids)

print(f"Deleted {len(window_sample_ids)} samples with Window predictions")

# Save the changes
dataset.save()

Model ready
                                                                             
0: 480x640 (no detections), 6.5ms
Speed: 0.7ms preprocess, 6.5ms inference, 0.5ms postprocess per image at shape (1, 3, 480, 640)

0: 640x416 (no detections), 6.5ms
Speed: 0.6ms preprocess, 6.5ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 416)
                                                                                    
0: 480x640 1 window, 6.8ms
Speed: 0.7ms preprocess, 6.8ms inference, 1.1ms postprocess per image at shape (1, 3, 480, 640)

0: 640x448 (no detections), 6.9ms
Speed: 0.8ms preprocess, 6.9ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 448)

0: 448x640 (no detections), 7.0ms
Speed: 0.7ms preprocess, 7.0ms inference, 0.5ms postprocess per image at shape (1, 3, 448, 640)

0: 480x640 (no detections), 7.9ms
Speed: 0.8ms preprocess, 7.9ms inference, 0.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 6.4ms
Speed: 1.4ms 

In [7]:


# Get class list
classes = dataset.default_classes

# Run the model on GPU if it is available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Load the YOLO model (not just the path)
model = YOLO("/mnt/EBI-SHARE/06_Data/labelstudio/models/lights/best.pt")
model.to(device)

print("Model ready")

# Add predictions to samples
with fo.ProgressBar() as pb:
    for sample in pb(dataset):
        # Load image as PIL Image (don't convert to tensor)
        image = Image.open(sample.filepath)
        
        # Perform inference directly on PIL image
        results = model(image, conf=0.4)
        
        # Extract predictions from results
        result = results[0]
        boxes = result.boxes.xyxy.cpu().numpy()  # x1, y1, x2, y2
        scores = result.boxes.conf.cpu().numpy()
        labels = result.boxes.cls.cpu().numpy().astype(int)
        
        # Get image dimensions
        w, h = image.size
        
        # Convert detections to FiftyOne format
        detections = []
        for label, score, box in zip(labels, scores, boxes):
            # Convert to [top-left-x, top-left-y, width, height]
            # in relative coordinates in [0, 1] x [0, 1]
            x1, y1, x2, y2 = box
            rel_box = [x1 / w, y1 / h, (x2 - x1) / w, (y2 - y1) / h]

            detections.append(
                fo.Detection(
                    label=model.names[label],  # Use model's class names
                    bounding_box=rel_box,
                    confidence=score
                )
            )

        # Save predictions to dataset
        sample["predictions"] = fo.Detections(detections=detections)
        sample.save()

# Filter samples that have "Light" predictions
light_predictions_view = dataset.filter_labels("predictions", fo.ViewField("label") == "light")

# Get the sample IDs that contain Light predictions
light_sample_ids = list(light_predictions_view.values("id"))

# Delete these samples from the dataset
dataset.delete_samples(light_sample_ids)

print(f"Deleted {len(light_sample_ids)} samples with Light predictions")

# Save the changes
dataset.save()

Model ready
                                                                             
0: 480x640 (no detections), 7.6ms
Speed: 0.9ms preprocess, 7.6ms inference, 0.5ms postprocess per image at shape (1, 3, 480, 640)
                                                                                    
0: 640x416 (no detections), 6.8ms
Speed: 0.6ms preprocess, 6.8ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 416)

0: 640x448 (no detections), 7.0ms
Speed: 0.7ms preprocess, 7.0ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 448)

0: 448x640 (no detections), 6.8ms
Speed: 0.7ms preprocess, 6.8ms inference, 0.5ms postprocess per image at shape (1, 3, 448, 640)

0: 480x640 (no detections), 6.6ms
Speed: 0.6ms preprocess, 6.6ms inference, 0.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 light, 6.4ms
Speed: 1.1ms preprocess, 6.4ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 640x544 (no detections), 6.7ms
Speed: 0.7ms p

In [8]:

print(dataset.count_sample_tags())
four.random_split(dataset, {"train": 0.7, "test": 0.2, "val": 0.1})
print(dataset.count_sample_tags())

{'train': 3890}
{'val': 389, 'train': 3890, 'test': 778}


In [4]:
# Visualize the dataset in the FiftyOne App

import fiftyone as fo

# Load existing dataset by name
dataset = fo.load_dataset("der_merger")

# Or list all available datasets first
print(fo.list_datasets())

# Then load the one you want
dataset = fo.load_dataset("der_merger")

DatasetNotFoundError: Dataset der_merger not found

In [5]:
import fiftyone as fo

# Start FiftyOne App without any dataset
session = fo.launch_app()

In [13]:


export_dir = "/mnt/EBI-SHARE/06_Data/labelstudio/datasets/test/rundgang/"
#label_field = "ground_truth"  # for example

# The splits to export
splits = ["train", "test","val"]

# All splits must use the same classes list
classes = ["chair"]

# Export the splits
for split in splits:
    split_view = dataset.match_tags(split)
    split_view.export(
        export_dir=export_dir,
        dataset_type=fo.types.YOLOv5Dataset,
        #label_field=label_field,
        split=split,
        classes=classes,
    )

Found multiple fields ['ground_truth', 'predictions'] with compatible type (<class 'fiftyone.core.labels.Detections'>, <class 'fiftyone.core.labels.Polylines'>); exporting 'ground_truth'


   0% ||--------------|    0/3890 [2.2s elapsed, ? remaining, ? samples/s] 

/home/freeze/miniconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'fork' not in provided classes
  warnings.warn(msg)
/home/freeze/miniconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'knife' not in provided classes
  warnings.warn(msg)
/home/freeze/miniconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'pizza' not in provided classes
  warnings.warn(msg)
/home/freeze/miniconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'dining table' not in provided classes
  warnings.warn(msg)
/home/freeze/miniconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'person' not in provided classes
  warnings.warn(msg)
/home/freeze/

                                                                           
Could not connect session, trying again in 10 seconds



/home/freeze/miniconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'cake' not in provided classes
  warnings.warn(msg)
/home/freeze/miniconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'bottle' not in provided classes
  warnings.warn(msg)
/home/freeze/miniconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'wine glass' not in provided classes
  warnings.warn(msg)
/home/freeze/miniconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'umbrella' not in provided classes
  warnings.warn(msg)
/home/freeze/miniconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'handbag' not in provided classes
  warnings.warn(msg)
/home/free

...

  46% |██████\--------| 1790/3890 [14.8s elapsed, 14.2s remaining, 198.7 samples/s] 

/home/freeze/miniconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'giraffe' not in provided classes
  warnings.warn(msg)


  54% |████████\------| 2114/3890 [16.4s elapsed, 11.2s remaining, 196.4 samples/s] 

/home/freeze/miniconda3/envs/onemanstreasure/lib/python3.10/site-packages/fiftyone/utils/yolo.py:1030: UserWarning: Ignoring object with label 'zebra' not in provided classes
  warnings.warn(msg)


 100% |███████████████| 3890/3890 [27.3s elapsed, 0s remaining, 198.1 samples/s]      
Directory '/mnt/EBI-SHARE/06_Data/labelstudio/datasets/test/rundgang/' already exists; export will be merged with existing files
Found multiple fields ['ground_truth', 'predictions'] with compatible type (<class 'fiftyone.core.labels.Detections'>, <class 'fiftyone.core.labels.Polylines'>); exporting 'ground_truth'
 100% |█████████████████| 778/778 [6.2s elapsed, 0s remaining, 194.7 samples/s]      
Directory '/mnt/EBI-SHARE/06_Data/labelstudio/datasets/test/rundgang/' already exists; export will be merged with existing files
Found multiple fields ['ground_truth', 'predictions'] with compatible type (<class 'fiftyone.core.labels.Detections'>, <class 'fiftyone.core.labels.Polylines'>); exporting 'ground_truth'
 100% |█████████████████| 389/389 [2.9s elapsed, 0s remaining, 202.5 samples/s]      
